In [3]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
## 데이터셋, 데이터로더 관련 모듈
from torch.nn.utils.rnn import pad_sequence
## 데이터 길이 맞추기

## 토커나이저, 단어사전 관련 모듈
from torchtext.data.utils import get_tokenizer              ## 토커나이저 인스턴스 추출
from torchtext.vocab import build_vocab_from_iterator       ## 데이터셋에서 단어사전 생성 함수
from nltk.corpus import stopwords                           ## 불용어 데이터셋
import nltk
from nltk.tokenize import word_tokenize

c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\data\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\vocab\__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
c:\Users\kdt\anaconda3\envs\NLP\lib\site-packages\torchtext\utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATU

In [29]:
FILE_DIR = './data/'
FILE_PATH = FILE_DIR+'IMDb_Reviews.csv'


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
import string

STOPWORDS = stopwords.words('english')
TOKENIZER = get_tokenizer('basic_english')
PUNC = string.punctuation
PUNC = PUNC+''.join([str(x) for x in range(10)])

In [6]:
dataDF= pd.read_csv(FILE_PATH)
texts = dataDF['review'].tolist()
labels = dataDF['sentiment'].tolist()
dataDF.head()

,review,sentiment
0,My family and I normally do not watch local mo...,1
1,"Believe it or not, this was at one time the wo...",0
2,"After some internet surfing, I found the ""Home...",0
3,One of the most unheralded great works of anim...,1
4,"It was the Sixties, and anyone with long hair ...",0


In [7]:
def yield_tokens(data):
    for line in data:
        line = ''.join([x for x in line if x not in PUNC])
        yield word_tokenize(line.lower())


In [8]:
VOCAB = build_vocab_from_iterator(yield_tokens(texts), specials=["<unk>", "<pad>"])
VOCAB.set_default_index(VOCAB["<unk>"])

In [9]:
def encode_texts(data, vocab):
    encoded = []
    for line in data:
        line = ''.join([c for c in line if c not in PUNC])
        tokens = word_tokenize(line.lower())
        token_ids = [vocab[token] for token in tokens]
        encoded.append(torch.tensor(token_ids, dtype=torch.long))
    return encoded

In [10]:
encoded_sequences = encode_texts(texts, VOCAB)

padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=VOCAB["<pad>"])
labels_tensor = torch.tensor(labels, dtype=torch.long)


In [11]:
class customDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

In [12]:
from sklearn.model_selection import train_test_split

In [13]:
X_train, X_temp, y_train, y_temp = train_test_split(
    padded_sequences, labels_tensor, test_size=0.2, random_state=42, stratify=labels_tensor
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [14]:
train_dataset = customDataset(X_train, y_train)
val_dataset = customDataset(X_val, y_val)
test_dataset = customDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=100, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [ ]:
## ----------------------------------------------------------------------
## 함수기능 : 배치크기 만큼 데이터셋 로딩해서 토큰 + 텐서화 진행 후 반환
## ----------------------------------------------------------------------
def collate_batch(batch):
    ## 라벨, 뉴스, 뉴스기사 시작 위치값 저장 변수
    label_list, news_list, offsets = [], [], [0]

    ## 1개씩 라벨과 뉴스 기사 추출
    for label, news in batch:
        ## 라벨 인코딩 후 추가 : 1 ~ 4 => 0 ~ 3
        label_list.append(label_pipeline(label))

        ## 뉴스 기사 인코딩 후 추가 
        processed_news = torch.tensor(text_pipeline(news), dtype=torch.int64)
        news_list.append(processed_news)

        ## 다음 뉴스를 읽기 위한 위치값 정보
        offsets.append(processed_news.size(0))
        #print(f'news 토큰 수 => {processed_news.size(0)}개')

    ## 배치 크기 만큼의 라벨 리스트 => 텐서화
    label_list = torch.tensor(label_list, dtype=torch.int64)

    ## 배치 크기 만큼의 길이 위치값 => 텐서화 
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    ## 배치 크기 만큼의 뉴스 기사 리스트 => 텐서화 
    news_list = torch.cat(news_list)

    return label_list.to(DEVICE), news_list.to(DEVICE), offsets.to(DEVICE)


In [15]:
len(train_dataset)

40000

In [16]:
# VCOAB_SIZE = V
len(VOCAB)

176669

In [17]:
train_dataset[0]

(tensor([48, 26, 75,  ...,  1,  1,  1]), tensor(1))

In [18]:
import torch.optim as optim 
from torch.optim.lr_scheduler import StepLR

DNN Model  with embedding

In [19]:
## -------------------------------------------------------------------------
## 클래스이름 : TextModel
## 부모클래스 : Module
## 매개변수둘 : 단어사전 갯수, 임베딩 수, 2진분류
## -------------------------------------------------------------------------
class TextDnnEmbModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(TextDnnEmbModel, self).__init__()
        
        # 1. EmbeddingBag: 평균 임베딩을 자동 계산
        self.embedding_bag = nn.EmbeddingBag(vocab_size, embedding_dim, mode='mean')
        
        # 2. DNN 분류기
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)  # 이진 분류용
        )

    def forward(self, text, offsets):
        # text: 모든 샘플을 1D로 펼친 단어 인덱스 (LongTensor)
        # offsets: 각 문장의 시작 위치를 나타내는 텐서 (LongTensor)
        
        embedded = self.embedding_bag(text, offsets)  # (batch_size, embedding_dim)
        return self.classifier(embedded)              # (batch_size, 1)

    ## 이진분류이므로 sigmod지만 
    ## BCEWithLogitsLoss 쓸것.


In [20]:
## 학습 설정
INPUT_SIZE      = 2460
LR              = 0.01
EPOCHS          = 10
STEP_SIZE       = 5
NUM_CLASS       = 1

EMBEDDING_DIM   = 256
HIDDEN_DIM      = 128
VOCAB_SIZE      = len(VOCAB)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
## 학습 인스턴스 생성
MODEL = TextDnnEmbModel(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
MODEL.to(DEVICE)

LOSS_FN   = nn.BCEWithLogitsLoss()
OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)
SCHEDULER = StepLR(OPTIMIZER, STEP_SIZE, gamma=0.1)
""" 
Decays the learning rate of each parameter group by gamma every step_size epochs. 
Notice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. 
When last_epoch=-1, sets initial lr as lr.

"""
## 전에 쓴 건 스코어가 변하지 않으면 patience만큼 기다렸다가 학습중지


' \nDecays the learning rate of each parameter group by gamma every step_size epochs. \nNotice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. \nWhen last_epoch=-1, sets initial lr as lr.\n\n'

In [23]:
for idx, (label, text) in enumerate(train_loader):
    print( idx, label.shape, text.shape)
    break

0 torch.Size([100, 2460]) torch.Size([100])


In [24]:
## -------------------------------------------------------------------------
## 함수기능 : 학습데이터를 사용하여 학습 진행
## 함수이름 : training
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def training(dataloader):
    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text)

        loss = LOSS_FN(pre, label.reshape(-1,1).float())
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1



In [25]:
## -------------------------------------------------------------------------
## 함수기능 : 검증데이터를 사용하여 학습 진행
## 함수이름 : evaluate
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def evaluate(dataloader):
    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text)
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1,1).float())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [26]:
## 모델 및 모델 층별 상태값 즉, 파라미터 값 저장 경로
MODEL_DIR  = './models/'
MODEL_FILE = 'IMDB_DNN_MODEL.pt'

In [27]:
EPOCHS = 100  ## 임시
# 모델 저장 기준
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(train_loader)
    valid_loss, valid_acc = evaluate(val_loader)
    SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print("-" * 59)

    ## 모델 저장 
    if MAX_ACC < valid_acc : 
        torch.save(MODEL, MODEL_DIR+MODEL_FILE)
        MAX_ACC = valid_acc


TypeError: forward() missing 1 required positional argument: 'offsets'

#### RNN Model

In [ ]:
## -------------------------------------------------------------------------
## 클래스이름 : TextModel
## 부모클래스 : Module
## 매개변수둘 : 단어사전 갯수, 임베딩 수, 2진분류
## -------------------------------------------------------------------------
class TextRnnModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim=1, num_layers=1):
        super(TextRnnModel, self).__init__()
        # RNN 레이어
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True)
        # RNN의 마지막 출력 이후, Linear 레이어로 이진 분류
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: (batch_size, seq_len, input_dim)
        # RNN에 입력 (hidden state와 cell state는 기본값으로 초기화)
        rnn_out, _ = self.rnn(x.float())  # (batch_size, seq_len, hidden_dim)
        # RNN의 마지막 출력을 가져와서 분류
        last_hidden_state = rnn_out
        # 마지막 출력에서 예측
        output = self.fc(last_hidden_state)  # (batch_size, output_dim)
        return output

    ## 이진분류이므로 sigmod지만 
    ## BCEWithLogitsLoss 쓸것.


In [ ]:
## 학습 설정
INPUT_SIZE      = 2460
HIDDEN_DIM      = 128
LR              = 0.01
EPOCHS          = 10
STEP_SIZE       = 5

NUM_CLASS       = 1
VOCAB_SIZE      = len(VOCAB)


In [ ]:
## 학습 인스턴스 생성
MODEL = TextRnnModel(INPUT_SIZE, HIDDEN_DIM)
MODEL.to(DEVICE)

LOSS_FN   = nn.BCEWithLogitsLoss()
OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)
SCHEDULER = StepLR(OPTIMIZER, STEP_SIZE, gamma=0.1)
""" 
Decays the learning rate of each parameter group by gamma every step_size epochs. 
Notice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. 
When last_epoch=-1, sets initial lr as lr.

"""
## 전에 쓴 건 스코어가 변하지 않으면 patience만큼 기다렸다가 학습중지


' \nDecays the learning rate of each parameter group by gamma every step_size epochs. \nNotice that such decay can happen simultaneously with other changes to the learning rate from outside this scheduler. \nWhen last_epoch=-1, sets initial lr as lr.\n\n'

In [ ]:
for idx, (label, text) in enumerate(train_loader):
    print( idx, label.shape, text.shape)
    break

0 torch.Size([100, 2460]) torch.Size([100])


In [ ]:
## -------------------------------------------------------------------------
## 함수기능 : 학습데이터를 사용하여 학습 진행
## 함수이름 : training
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def training(dataloader):
    ## 학습 모드 설정
    MODEL.train()

    ## 학습 손실과 점수 저장 
    total_loss, total_acc = 0, 0
    
    for idx, (text, label) in enumerate(dataloader):

        OPTIMIZER.zero_grad()
        pre  = MODEL(text)

        loss = LOSS_FN(pre, label.reshape(-1,1).float())
        loss.backward()
        ## gradient vanishing, gradient exploding 발생 => 방지 및 안정화 
        ## - gradient가 일정 threshold를 넘어가면 clipping
        ## - clipping: gradient의 L2norm(norm이지만 보통 L2 norm사용)으로 나눠주는 방식
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 0.1)
        OPTIMIZER.step()

        total_loss += loss.item()
        total_acc += (pre.argmax(dim=1) == label).sum().item()

        if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1



In [ ]:
## -------------------------------------------------------------------------
## 함수기능 : 검증데이터를 사용하여 학습 진행
## 함수이름 : evaluate
## 매개변수 : 데이터로더
## 결과반환 : 손실값, 모델성능값
## -------------------------------------------------------------------------
def evaluate(dataloader):
    MODEL.eval()
    total_loss, total_acc = 0, 0

    with torch.no_grad():
        for idx, (text, label) in enumerate(dataloader):
            ## 추론 진행
            pre = MODEL(text)
            ## 손실 계산
            loss = LOSS_FN(pre, label.reshape(-1,1).float())

            ## 손실 및 성능 평가
            total_loss += loss.item()
            total_acc += (pre.argmax(1) == label).sum().item()
            if idx==5: break
        
    return total_loss/idx+1, total_acc/idx+1

In [ ]:
## 모델 및 모델 층별 상태값 즉, 파라미터 값 저장 경로
MODEL_DIR  = './models/'
MODEL_FILE = 'IMDB_RNN_MODEL.pt'

In [ ]:
EPOCHS = 100  ## 임시
# 모델 저장 기준
MAX_ACC = 0.

for epoch in range(1, EPOCHS + 1):
    
    train_loss, train_acc = training(train_loader)
    valid_loss, valid_acc = evaluate(val_loader)
    SCHEDULER.step()

    print("-" * 59)
    print(f'| end of epoch {epoch:3d} | train acc {train_acc:8.3f}  | valid acc {valid_acc:8.3f}')
    print("-" * 59)

    ## 모델 저장 
    if MAX_ACC < valid_acc : 
        torch.save(MODEL, MODEL_DIR+MODEL_FILE)
        MAX_ACC = valid_acc


-----------------------------------------------------------
| end of epoch   1 | train acc   56.200  | valid acc   57.800
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   2 | train acc   59.600  | valid acc   57.800
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   3 | train acc   64.000  | valid acc   57.800
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   4 | train acc   61.200  | valid acc   57.800
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   5 | train acc   61.400  | valid acc   57.800
-----------------------------------------------------------
-----------------------------------------------------------
| end of epoch   6 | train acc